# Análise reproduzível de Boletins de Urna — TSE 2026

**Objetivo:** calcular tabelas e gráficos a partir dos BUs presidenciais do segundo turno e comparar resultados com valores de referência configurados, sem presumir que esses valores sejam fatos confirmados.

**Uso no Colab:** confira os parâmetros abaixo e escolha **Ambiente de execução → Executar tudo**. O padrão é um **piloto no Acre**. Para o conjunto nacional, mude `ESCOPO` para `"COMPLETO"`; o exterior será incluído. CPU é suficiente. Não conecte o Google Drive: os dados vêm de fontes públicas. Baixe o ZIP ao final, pois o armazenamento do Colab é temporário.

**Fonte:** [Boletins de Urna oficiais do TSE](https://dadosabertos.tse.jus.br/dataset/resultados-2026-boletim-de-urna). A consulta utiliza a versão disponível no catálogo no momento da execução. URLs e hashes são preservados para comparação entre execuções.

**Limites:** o BU contém totais finais por seção; não revela a hora de cada voto por candidato. O corte inclusivo escolhido é `17:16:00`, em Brasília, sob a hipótese de fuso configurada. O cálculo temporal mantém todas as seções e fornece limites, preservando resultados indeterminados. Correlações e Benford não estabelecem fraude.

Esta cópia é distribuída sem saídas, referências a contas ou histórico de sessão. Seus resultados serão calculados no seu ambiente. A validação das fontes e do empacotamento está descrita no guia do pacote; uma execução piloto não valida cobertura nacional.


**Primeiro acesso:** leia a [apresentação e o roteiro de download das bases](https://github.com/antunesclarissa/tse-reproducivel#1-de-onde-vêm-os-dados-e-onde-baixar-as-bases) e as [definições e limitações](https://github.com/antunesclarissa/tse-reproducivel/blob/main/METODOLOGIA.md). O programa baixa os dados automaticamente; o download manual serve para inspecionar as fontes.


## 1. Parâmetros

Altere aqui escopo, UFs do piloto, corte inclusivo e hipótese de fuso. `HORARIO_CORTE` afeta a tabela de limites; as faixas históricas 17h/17h16/19h dos gráficos comparativos são fixas e continuam identificadas pelos próprios rótulos. A versão é restrita ao segundo turno presidencial de 2026.

In [ ]:
from pathlib import Path
ESCOPO = "PILOTO" # @param ["COMPLETO", "PILOTO"]
INSTALAR_DEPENDENCIAS = True
DATA_ELEICAO = "2026-10-25"
HORARIO_CORTE = "17:16:00" # @param {type:"string"}
ANO = 2026 # @param {type:"integer"}
TURNO = 2 # @param {type:"integer"}
CARGO = 1  # Presidente; confira o dicionário se alterar.
UF_PILOTO = ["AC"]
INCLUIR_EXTERIOR = True # @param {type:"boolean"}
EXIGIR_TODAS_PARTICOES = True # @param {type:"boolean"}
TAMANHO_BLOCO = 200_000
SEMENTE = 42
MINIMOS_VOTOS_VALIDOS = [1, 50, 100, 200]
MINIMO_SECOES_CORRELACAO = 10
LIMIAR_SUPERMAIORIA = 0.90
MINIMO_BENFORD = 100
# Hipótese explícita sobre o relógio gravado. As duas versões permanecem exportadas.
RELOGIO_ARQUIVO = "LOCAL" # @param ["LOCAL", "BRASILIA", "DESCONHECIDO"]
# Opcional: totais INDEPENDENTES do resultado oficial, com URL de origem.
# Ex.: {"fonte": "https://...", "votos_por_numero": {"13": 123, "22": 456}}
# Deixe vazio para registrar que a reconciliação externa ainda está pendente.
REFERENCIA_OFICIAL = {}
RAIZ = Path("/content/tse_reproducivel") if Path("/content").exists() else Path("resultados_tse_2026")
RAIZ = RAIZ.resolve()
ALEGACOES = {
    "secoes": 499205, "votos_apurados_termo_ambiguo": 128236074,
    "contador_biometrico": 8632488, "margem_absoluta": 4430191,
    "participacao_flavio_ate17": 0.5239,
    "unanimes_lula": 106, "unanimes_flavio": 4,
    "acima90_lula": 6899, "acima90_flavio": 384,
}
assert ANO == 2026 and TURNO == 2 and CARGO == 1, "Esta versão foi verificada para presidente, 2º turno de 2026."
assert ESCOPO in {"COMPLETO", "PILOTO"}
assert RELOGIO_ARQUIVO in {"LOCAL", "BRASILIA", "DESCONHECIDO"}


### Preparar as bibliotecas

No Colab, instala apenas as dependências ausentes. Pelo comando, use antes `requirements.txt` no ambiente Python escolhido.

In [ ]:
# Instalação apenas das dependências ausentes; versões efetivas serão registradas.
import importlib.util, subprocess, sys
DEPENDENCIAS = {
    "pandas": "pandas>=2.2,<4", "numpy": "numpy>=1.26,<3",
    "matplotlib": "matplotlib>=3.8,<4", "scipy": "scipy>=1.12,<2",
    "pyarrow": "pyarrow>=15,<27", "requests": "requests>=2.31,<3",
}
faltantes = [req for mod, req in DEPENDENCIAS.items() if importlib.util.find_spec(mod) is None]
if faltantes and not INSTALAR_DEPENDENCIAS:
    raise RuntimeError("Dependências ausentes; instale requirements.txt no ambiente atual.")
if faltantes:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *faltantes])
print("Dependências disponíveis.")

## 2. Funções de aquisição e integridade
O catálogo é consultado a cada execução. O ZIP é verificado com o SHA-512 oficial antes da leitura. Um hash confirma a correspondência com o arquivo publicado; não prova a exatidão substantiva de seus dados.
O dicionário PDF de cada arquivo é preservado, sem executar conteúdo dos documentos.

In [ ]:
import datetime as dt
import hashlib, html, io, json, platform, re, shutil, unicodedata, warnings, zipfile
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from scipy import stats
from IPython.display import display, Markdown
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

np.random.seed(SEMENTE)
CATALOGO = f"https://dadosabertos.tse.jus.br/dataset/resultados-{ANO}-boletim-de-urna"
API = "https://dadosabertos.tse.jus.br/api/3/action/package_show"
UFS_BR = "AC AL AM AP BA CE DF ES GO MA MG MS MT PA PB PE PI PR RJ RN RO RR RS SC SE SP TO".split()
UF_ESPERADAS = UFS_BR + (["ZZ"] if INCLUIR_EXTERIOR else []) if ESCOPO == "COMPLETO" else list(UF_PILOTO)
assert set(UF_ESPERADAS) <= set(UFS_BR + ["ZZ"])
CACHE = RAIZ / "arquivos"
CACHE.mkdir(parents=True, exist_ok=True)
# Uma pasta nova por execução impede misturar tabelas de piloto e execução nacional.
RAIZ = RAIZ / "execucoes" / (dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%S%fZ") + "_" + ESCOPO.lower())
for pasta in ["dicionarios", "tabelas", "graficos", "intermediarios"]:
    (RAIZ / pasta).mkdir(parents=True, exist_ok=True)

sessao = requests.Session()
sessao.headers.update({"User-Agent": "TSE-reproducao-academica/1.0"})
sessao.mount("https://", HTTPAdapter(max_retries=Retry(total=3, backoff_factor=1, status_forcelist=[429, 500, 502, 503, 504])))
def agora():
    return dt.datetime.now(dt.timezone.utc).isoformat()
def json_save(nome, obj):
    (RAIZ / nome).write_text(json.dumps(obj, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
def sha512(path):
    h = hashlib.sha512()
    with open(path, "rb") as f:
        for bloco in iter(lambda: f.read(1024 * 1024), b""):
            h.update(bloco)
    return h.hexdigest()
def descobrir_recursos():
    try:
        # Evita resposta antiga de cache, observada em uma consulta feita pelo Colab.
        r = sessao.get(API, params={"id": f"resultados-{ANO}-boletim-de-urna", "_": int(dt.datetime.now().timestamp())}, headers={"Cache-Control": "no-cache"}, timeout=(15, 90))
        r.raise_for_status()
        resposta = r.json()
        if not resposta.get("success"):
            raise RuntimeError("Catálogo CKAN sem sucesso")
        pacote = resposta["result"]
        json_save("catalogo_consultado.json", resposta)
        recursos = pacote["resources"]
    except (requests.RequestException, ValueError, KeyError, RuntimeError) as erro:
        print("API indisponível; consultando links públicos do mesmo catálogo:", type(erro).__name__)
        r = sessao.get(CATALOGO, params={"_": int(dt.datetime.now().timestamp())}, headers={"Cache-Control": "no-cache"}, timeout=(15, 90)); r.raise_for_status()
        (RAIZ / "catalogo_consultado.html").write_text(r.text, encoding="utf-8")
        links = re.findall(r'href=[\"\x27]([^\"\x27]+)[\"\x27]', r.text)
        recursos = [{"url": html.unescape(x), "name": ""} for x in links if "bweb_" in x]
    candidatos = []
    for recurso in recursos:
        url = recurso.get("url", "")
        url_catalogo = url
        # O recurso CSV de RR foi observado apontando para o próprio .sha512.
        # Só resolve o ZIP se seu nome estiver explicitamente listado no hash oficial.
        if "Boletim de Urna" in recurso.get("name", "") and url.endswith(".zip.sha512"):
            resposta_hash = sessao.get(url, timeout=(15, 90)); resposta_hash.raise_for_status()
            nome_zip = url.rsplit("/", 1)[-1].removesuffix(".sha512")
            if nome_zip not in resposta_hash.text:
                raise RuntimeError("O hash não identifica o ZIP anunciado pelo recurso.")
            url = url.removesuffix(".sha512")
            print("Link do catálogo resolvido pelo nome publicado no hash:", nome_zip)
        # Descoberta pelo catálogo; não adivinha o identificador da eleição nem a data de geração.
        m = re.search(rf"/bweb_{TURNO}t_([A-Z]{{2}})_(\d{{12}})\.zip$", url)
        if m and m.group(1) in UF_ESPERADAS:
            candidatos.append({"uf": m.group(1), "versao": m.group(2), "url": url, "url_original_catalogo": url_catalogo, "recurso_id": recurso.get("id"), "nome_catalogo": recurso.get("name")})
    if not candidatos:
        raise RuntimeError("Nenhum recurso compatível encontrado no catálogo oficial.")
    selecionados = []
    for uf in UF_ESPERADAS:
        opcoes = [x for x in candidatos if x["uf"] == uf]
        if not opcoes:
            continue
        # Datas no nome são DDMMYYYYHHMM.
        item = max(opcoes, key=lambda x: dt.datetime.strptime(x["versao"], "%d%m%Y%H%M"))
        hashes = [r["url"] for r in recursos if r.get("url") == item["url"] + ".sha512"]
        if not hashes:
            raise RuntimeError(f"{uf}: hash correspondente ausente no catálogo.")
        selecionados.append({**item, "url_hash": hashes[0]})
    json_save("recursos_selecionados.json", selecionados)
    ausentes = sorted(set(UF_ESPERADAS) - {x["uf"] for x in selecionados})
    if ausentes and EXIGIR_TODAS_PARTICOES:
        raise RuntimeError(f"Partições ausentes no catálogo: {ausentes}")
    return selecionados

def baixar_validado(recurso):
    caminho = CACHE / recurso["url"].rsplit("/", 1)[-1]
    r = sessao.get(recurso["url_hash"], timeout=(15, 90)); r.raise_for_status()
    texto_hash = r.text
    (CACHE / (caminho.name + ".sha512")).write_text(texto_hash, encoding="utf-8")
    encontrados = re.findall(r"(?<![0-9a-fA-F])[0-9a-fA-F]{128}(?![0-9a-fA-F])", texto_hash)
    if len(encontrados) != 1:
        raise RuntimeError(f"{recurso['uf']}: formato de hash ambíguo.")
    esperado = encontrados[0].lower()
    cache = caminho.exists() and sha512(caminho) == esperado
    if not cache:
        parcial = caminho.with_suffix(".zip.part")
        with sessao.get(recurso["url"], stream=True, timeout=(15, 180)) as resposta:
            resposta.raise_for_status()
            with open(parcial, "wb") as f:
                for bloco in resposta.iter_content(1024 * 1024):
                    f.write(bloco)
        if sha512(parcial) != esperado:
            parcial.unlink(missing_ok=True)
            raise RuntimeError(f"{recurso['uf']}: SHA-512 divergente; processamento interrompido.")
        parcial.replace(caminho)
    return caminho, {**recurso, "arquivo": caminho.name, "bytes": caminho.stat().st_size, "sha512": esperado, "hash_verificado": True, "verificado_em_utc": agora(), "cache": cache}

## 3. Leitura e consolidação por seção
O dicionário oficial define `QT_ELEI_BIOM_SEM_HABILITACAO` como a quantidade de eleitores com biometria que não foram habilitados por meio dela. A taxa abaixo usa **comparecimento** como denominador e deve ser lida dessa forma; seu complemento não mede a taxa de sucesso da biometria entre os cadastrados.

Os contadores da seção repetem-se nas linhas dos candidatos. O código verifica sua consistência e usa uma única ocorrência. Seções agregadas não são expandidas nem contadas duas vezes. Registros conflitantes interrompem a análise, em vez de escolher arbitrariamente uma versão.

In [ ]:
CHAVE = ["ANO_ELEICAO", "NR_TURNO", "CD_ELEICAO", "SG_UF", "CD_MUNICIPIO", "NR_ZONA", "NR_SECAO", "CD_CARGO_PERGUNTA"]
META = ["NM_MUNICIPIO", "QT_APTOS", "QT_COMPARECIMENTO", "QT_ABSTENCOES", "DT_ABERTURA", "DT_ENCERRAMENTO", "QT_ELEI_BIOM_SEM_HABILITACAO", "NR_URNA_EFETIVADA", "DS_SECOES_AGREGADAS"]
VOTO = ["CD_TIPO_URNA", "DS_TIPO_VOTAVEL", "CD_TIPO_VOTAVEL", "NR_VOTAVEL", "NM_VOTAVEL", "QT_VOTOS"]
NUM = ["QT_APTOS", "QT_COMPARECIMENTO", "QT_ABSTENCOES", "QT_ELEI_BIOM_SEM_HABILITACAO", "QT_VOTOS"]

def ler_uf(caminho, uf):
    partes, descartes = [], {}
    with zipfile.ZipFile(caminho) as z:
        for nome in z.namelist():
            if nome.lower().endswith(".pdf"):
                destino = RAIZ / "dicionarios" / f"{uf}_{Path(nome).name}"
                destino.write_bytes(z.read(nome))
        csvs = [n for n in z.namelist() if n.lower().endswith(".csv")]
        if len(csvs) != 1:
            raise RuntimeError(f"{uf}: esperado um CSV por recurso; encontrados {len(csvs)}")
        with z.open(csvs[0]) as f:
            leitor = pd.read_csv(f, sep=";", encoding="latin1", dtype="string", chunksize=TAMANHO_BLOCO,
                usecols=CHAVE + META + VOTO, na_values=["#NULO#", "#NULO", "#NE#", "#NE"], keep_default_na=True)
            for bloco in leitor:
                filtro = bloco.ANO_ELEICAO.eq(str(ANO)) & bloco.NR_TURNO.eq(str(TURNO)) & bloco.CD_CARGO_PERGUNTA.eq(str(CARGO))
                b = bloco.loc[filtro].copy()
                if b.empty:
                    continue
                if not b.SG_UF.eq(uf).all():
                    raise RuntimeError(f"{uf}: registro com UF incompatível com a partição.")
                for tipo, n in b.CD_TIPO_URNA.value_counts(dropna=False).items():
                    descartes[str(tipo)] = descartes.get(str(tipo), 0) + int(n)
                b = b[b.CD_TIPO_URNA.eq("1")].copy()  # Apurada, conforme o dicionário.
                for c in NUM:
                    valor = pd.to_numeric(b[c], errors="raise")
                    b[c] = valor.mask(valor.isin([-1, -3])).astype("Int64")
                if b.QT_VOTOS.isna().any() or b.QT_VOTOS.lt(0).any():
                    raise RuntimeError(f"{uf}: votos ausentes ou negativos.")
                if b[CHAVE].isna().any().any():
                    raise RuntimeError(f"{uf}: identificador incompleto de seção.")
                partes.append(b)
    if not partes:
        raise RuntimeError(f"{uf}: nenhum registro apurado no recorte.")
    dados = pd.concat(partes, ignore_index=True)
    n_original = len(dados)
    dados = dados.drop_duplicates()
    identidade_voto = CHAVE + ["CD_TIPO_VOTAVEL", "NR_VOTAVEL"]
    conflito = dados.duplicated(identidade_voto, keep=False)
    if conflito.any():
        dados.loc[conflito].to_csv(RAIZ / "tabelas" / f"conflitos_{uf}.csv", index=False)
        raise RuntimeError(f"{uf}: linhas conflitantes por seção/votável. Veja o CSV de conflitos.")
    metadados = dados[CHAVE + META].drop_duplicates()
    if metadados.duplicated(CHAVE, keep=False).any():
        metadados.to_csv(RAIZ / "tabelas" / f"metadados_conflitantes_{uf}.csv", index=False)
        raise RuntimeError(f"{uf}: contadores ou metadados conflitantes dentro da seção.")
    tipos = dados.DS_TIPO_VOTAVEL.str.upper()
    desconhecido = ~tipos.isin(["NOMINAL", "BRANCO", "NULO"])
    if desconhecido.any():
        raise RuntimeError(f"{uf}: tipo votável não previsto: {dados.loc[desconhecido, 'DS_TIPO_VOTAVEL'].unique().tolist()}")
    total_tipo = dados.groupby(CHAVE + ["DS_TIPO_VOTAVEL"], dropna=False).QT_VOTOS.sum().unstack(fill_value=0)
    total_tipo.columns = ["votos_" + str(c).lower() for c in total_tipo.columns]
    secoes_uf = metadados.set_index(CHAVE).join(total_tipo).reset_index()
    for c in ["votos_nominal", "votos_branco", "votos_nulo"]:
        if c not in secoes_uf:
            secoes_uf[c] = 0
    secoes_uf = secoes_uf.rename(columns={"votos_nominal": "votos_validos", "votos_branco": "votos_brancos", "votos_nulo": "votos_nulos"})
    secoes_uf["votos_totais"] = secoes_uf[["votos_validos", "votos_brancos", "votos_nulos"]].sum(axis=1)
    nominal = dados.loc[tipos.eq("NOMINAL"), CHAVE + ["NR_VOTAVEL", "NM_VOTAVEL", "QT_VOTOS"]].copy()
    nominal.to_parquet(RAIZ / "intermediarios" / f"votos_{uf}.parquet", index=False)
    secoes_uf.to_parquet(RAIZ / "intermediarios" / f"secoes_{uf}.parquet", index=False)
    info = {"uf": uf, "linhas_apos_filtro": n_original, "duplicatas_identicas_removidas": n_original - len(dados), "secoes_com_bu": len(secoes_uf), "linhas_por_tipo_urna_antes_filtro": descartes}
    return secoes_uf, nominal, info

## 4. Baixar, verificar e processar
Esta célula pode levar vários minutos no modo completo. Downloads verificados são reutilizados em uma nova execução. Falhas são registradas; uma análise incompleta recebe aviso explícito.

In [ ]:
recursos = descobrir_recursos()
manifesto, qualidade, erros, lista_secoes, lista_votos = [], [], [], [], []
for recurso in recursos:
    uf = recurso["uf"]
    try:
        arquivo, registro = baixar_validado(recurso)
        s, v, q = ler_uf(arquivo, uf)
        manifesto.append(registro); qualidade.append(q)
        lista_secoes.append(s); lista_votos.append(v)
        print(f"{uf}: hash OK; {len(s):,} seções com BU presidencial apurado.")
    except Exception as erro:
        erros.append({"uf": uf, "erro": f"{type(erro).__name__}: {erro}"})
        print(f"{uf}: FALHOU — {erro}")
    json_save("manifesto_downloads.json", manifesto)
    json_save("qualidade_ingestao.json", qualidade)
    json_save("erros_ingestao.json", erros)
faltam = sorted(set(UF_ESPERADAS) - {r["uf"] for r in manifesto})
if faltam and EXIGIR_TODAS_PARTICOES:
    raise RuntimeError(f"Execução interrompida: faltam {faltam}. Corrija e execute novamente; arquivos íntegros serão reutilizados.")
if not lista_secoes:
    raise RuntimeError("Nenhuma partição foi processada com sucesso.")
secoes = pd.concat(lista_secoes, ignore_index=True)
votos = pd.concat(lista_votos, ignore_index=True)
del lista_secoes, lista_votos
if secoes.CD_ELEICAO.nunique() != 1 or secoes.duplicated(CHAVE).any():
    raise RuntimeError("Eleição ambígua ou chave de seção repetida.")
NACIONAL_COMPLETO = set(UFS_BR + ["ZZ"]) <= set(secoes.SG_UF.unique())
ROTULO = "Brasil + exterior, cobertura de partições completa" if NACIONAL_COMPLETO else "RECORTE PARCIAL: " + ", ".join(sorted(secoes.SG_UF.unique()))
display(Markdown(f"**{ROTULO}.** A cobertura de partições não garante, sozinha, a completude de todas as seções oficiais."))

## 5. Checagens contábeis e candidatos
A igualdade entre votos totais e comparecimento é uma checagem interna. A reconciliação com uma totalização oficial independente continua pendente enquanto `REFERENCIA_OFICIAL` estiver vazia.

In [ ]:
checagens = pd.DataFrame({
    "comparecimento_ausente": secoes.QT_COMPARECIMENTO.isna(),
    "votos_diferem_comparecimento": secoes.QT_COMPARECIMENTO.notna() & secoes.votos_totais.ne(secoes.QT_COMPARECIMENTO),
    "aptos_diferem_comparecimento_abstencao": secoes[["QT_APTOS", "QT_COMPARECIMENTO", "QT_ABSTENCOES"]].notna().all(axis=1) & secoes.QT_APTOS.ne(secoes.QT_COMPARECIMENTO + secoes.QT_ABSTENCOES),
    "biometria_ausente": secoes.QT_ELEI_BIOM_SEM_HABILITACAO.isna(),
    "biometria_fora_intervalo": secoes.QT_ELEI_BIOM_SEM_HABILITACAO.lt(0) | secoes.QT_ELEI_BIOM_SEM_HABILITACAO.gt(secoes.QT_COMPARECIMENTO),
})
checagens = checagens.fillna(False)
auditoria_contabil = checagens.sum().rename("secoes").to_frame()
display(auditoria_contabil)
auditoria_contabil.to_csv(RAIZ / "tabelas" / "checagens_contabeis.csv")
if checagens.any(axis=1).any():
    secoes.loc[checagens.any(axis=1)].assign(**checagens.loc[checagens.any(axis=1)].to_dict("series")).to_csv(RAIZ / "tabelas" / "secoes_com_alertas.csv", index=False)
if checagens[["comparecimento_ausente", "votos_diferem_comparecimento", "aptos_diferem_comparecimento_abstencao", "biometria_fora_intervalo"]].any().any():
    raise RuntimeError("Inconsistências contábeis materiais. Examine as seções com alertas antes de interpretar os resultados.")
mapa_nomes = votos.groupby("NR_VOTAVEL").NM_VOTAVEL.agg(lambda x: sorted(set(x.dropna())))
if not mapa_nomes.map(len).eq(1).all():
    raise RuntimeError("Nomes conflitantes para o mesmo número de candidato.")
CANDIDATOS = {str(k): v[0] for k, v in mapa_nomes.items()}
if len(CANDIDATOS) != 2:
    raise RuntimeError(f"Esperados dois candidatos no segundo turno: {CANDIDATOS}")
totais_candidatos = votos.groupby(["NR_VOTAVEL", "NM_VOTAVEL"]).QT_VOTOS.sum().reset_index(name="votos")
totais_candidatos["participacao_validos"] = totais_candidatos.votos / totais_candidatos.votos.sum()
display(totais_candidatos)
totais_candidatos.to_csv(RAIZ / "tabelas" / "candidatos.csv", index=False)
wide = votos.pivot(index=CHAVE, columns="NR_VOTAVEL", values="QT_VOTOS").fillna(0).astype("int64")
wide = wide.reindex(columns=list(CANDIDATOS), fill_value=0)
wide.columns = ["cand_" + c for c in wide.columns]
secoes = secoes.merge(wide.reset_index(), on=CHAVE, how="left", validate="one_to_one")
for numero in CANDIDATOS:
    # Ausência nominal numa seção de zero votos válidos é zero estrutural, não contador desconhecido.
    secoes["cand_" + numero] = secoes["cand_" + numero].fillna(0).astype("int64")
if not secoes[["cand_" + c for c in CANDIDATOS]].sum(axis=1).eq(secoes.votos_validos).all():
    raise RuntimeError("Votos dos candidatos não reconciliam com os válidos.")
if REFERENCIA_OFICIAL:
    if not REFERENCIA_OFICIAL.get("fonte", "").startswith("https://"):
        raise ValueError("Informe a URL da referência independente.")
    comparacao_oficial = []
    for numero, esperado in REFERENCIA_OFICIAL["votos_por_numero"].items():
        calculado = int(secoes["cand_" + str(numero)].sum())
        comparacao_oficial.append({"numero": numero, "oficial": esperado, "calculado": calculado, "diferenca": calculado - esperado, "mesmo_escopo_nacional": NACIONAL_COMPLETO})
    display(pd.DataFrame(comparacao_oficial))
    pd.DataFrame(comparacao_oficial).to_csv(RAIZ / "tabelas" / "reconciliacao_externa.csv", index=False)
else:
    print("Reconciliação externa com totalização independente: PENDENTE (referência não informada).")

## 6. Horários: bruto e normalizado
A conversão usa a referência oficial de 2026: [TSE](https://www.tse.jus.br/comunicacao/noticias/2026/Setembro/faltam-26-dias-votacao-comeca-e-termina-no-mesmo-horario-em-todo-o-pais) e [TRE-AM](https://www.tre-am.jus.br/comunicacao/noticias/2026/Setembro/eleitorado-do-amazonas-deve-ficar-atento-aos-horarios-de-votacao-nas-eleicoes-2026).
São 11 municípios do Amazonas no fuso do Acre. Fernando de Noronha tem ajuste próprio. A hipótese `LOCAL` é explícita: o dicionário descreve data/hora, mas não documenta o fuso da coluna. Confira também o resumo das aberturas. Se essa hipótese não for apropriada, altere o parâmetro; use `DESCONHECIDO` para não gerar a versão normalizada.
Encerrar após 17h não estabelece irregularidade nem distingue filas, eventos técnicos ou prorrogações. Exterior não é comparado ao corte simultâneo de Brasília.

In [ ]:
def normalizar_nome(x):
    return "".join(c for c in unicodedata.normalize("NFKD", str(x).upper()) if not unicodedata.combining(c))
AM_ACRE = {normalizar_nome(x) for x in ["Amaturá", "Atalaia do Norte", "Benjamin Constant", "Eirunepé", "Envira", "Guajará", "Ipixuna", "Itamarati", "Jutaí", "Tabatinga", "São Paulo de Olivença"]}
nomes_municipios = secoes.NM_MUNICIPIO.map(normalizar_nome)
ajustes = secoes.SG_UF.map({"AC": 2, "AM": 1, "RO": 1, "RR": 1, "MT": 1, "MS": 1}).fillna(0).to_numpy(dtype=float, copy=True)
ajustes[(secoes.SG_UF.eq("AM") & nomes_municipios.isin(AM_ACRE)).to_numpy(dtype=bool)] = 2
ajustes[(secoes.SG_UF.eq("PE") & nomes_municipios.str.contains("FERNANDO DE NORONHA")).to_numpy(dtype=bool)] = -1
ajustes[secoes.SG_UF.eq("ZZ").to_numpy(dtype=bool)] = np.nan
secoes["ajuste_brasilia_horas"] = ajustes
for origem, destino in [("DT_ENCERRAMENTO", "encerramento_arquivo"), ("DT_ABERTURA", "abertura_arquivo")]:
    secoes[destino] = pd.to_datetime(secoes[origem], format="%Y-%m-%d %H:%M:%S", errors="coerce")
invalidos_horario = secoes.encerramento_arquivo.isna()
secoes.loc[invalidos_horario, CHAVE + ["DT_ENCERRAMENTO"]].to_csv(RAIZ / "tabelas" / "horarios_ausentes_invalidos.csv", index=False)
def segundos_desde_corte(serie):
    # Preserva a data: encerramento no dia seguinte não é tratado como antes das 17h.
    corte = pd.Timestamp(f"{DATA_ELEICAO} 17:00:00")
    return (serie - corte).dt.total_seconds()
secoes["segundos_arquivo"] = segundos_desde_corte(secoes.encerramento_arquivo)
if RELOGIO_ARQUIVO == "LOCAL":
    ajuste = pd.to_timedelta(secoes.ajuste_brasilia_horas.astype(float), unit="h")
    secoes["encerramento_brasilia"] = secoes.encerramento_arquivo + ajuste
elif RELOGIO_ARQUIVO == "BRASILIA":
    secoes["encerramento_brasilia"] = secoes.encerramento_arquivo.where(~secoes.SG_UF.eq("ZZ"))
else:
    secoes["encerramento_brasilia"] = pd.NaT
secoes["segundos_brasilia"] = segundos_desde_corte(secoes.encerramento_brasilia)
def faixa_horario(segundos):
    labels = ["Até 17:00:00", ">17:00:00 a <17:16:00", "17:16:00 a <19:00:00", "A partir de 19:00:00"]
    arr = segundos.to_numpy(dtype=float, na_value=np.nan)
    valores = np.select([arr <= 0, (arr > 0) & (arr < 960), (arr >= 960) & (arr < 7200), arr >= 7200], labels, default="Horário ausente")
    return pd.Series(pd.Categorical(valores, categories=labels + ["Horário ausente"], ordered=True), index=segundos.index)
secoes["faixa_arquivo"] = faixa_horario(secoes.segundos_arquivo)
secoes["faixa_brasilia"] = faixa_horario(secoes.segundos_brasilia)
resumo_aberturas = secoes.assign(hora_abertura=secoes.abertura_arquivo.dt.strftime("%H")).groupby(["SG_UF", "ajuste_brasilia_horas", "hora_abertura"], dropna=False).size().rename("secoes").reset_index()
display(resumo_aberturas.head(40))
resumo_aberturas.to_csv(RAIZ / "tabelas" / "aberturas_para_conferencia_fusos.csv", index=False)

## 7. Tabelas e gráficos de encerramento
Cada linha soma **todos os votos das seções daquele grupo**, inclusive votos registrados antes do horário de corte. Não são votos individualmente datados. Percentuais são razões entre somas, nunca a média dos percentuais das seções.

In [ ]:
COLS_SOMA = ["QT_COMPARECIMENTO", "votos_validos", "votos_brancos", "votos_nulos", "QT_ELEI_BIOM_SEM_HABILITACAO"] + ["cand_" + c for c in CANDIDATOS]
CORES = dict(zip(CANDIDATOS, ["#2364AA", "#B66A16"]))
plt.rcParams.update({"figure.figsize": (10, 5.5), "font.size": 11, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": False, "savefig.bbox": "tight"})
def salvar_fig(nome):
    plt.tight_layout(); plt.savefig(RAIZ / "graficos" / (nome + ".png"), dpi=160); plt.show(); plt.close()
def agregar(base, grupos):
    g = base.groupby(grupos, observed=True, dropna=False)
    tab = g[COLS_SOMA].sum(min_count=1)
    tab.insert(0, "secoes", g.size())
    tab["secoes_biometria_conhecida"] = g.QT_ELEI_BIOM_SEM_HABILITACAO.count()
    for n in CANDIDATOS:
        tab["participacao_" + n] = tab["cand_" + n] / tab.votos_validos.replace(0, np.nan)
    a, b = list(CANDIDATOS)
    tab[f"margem_{a}_menos_{b}"] = tab["cand_" + a] - tab["cand_" + b]
    return tab.reset_index()
tabelas_horario = {}
cenarios = [("arquivo_todo_recorte", secoes, "faixa_arquivo"), ("exterior_relogio_local", secoes[secoes.SG_UF.eq("ZZ")], "faixa_arquivo")]
if RELOGIO_ARQUIVO != "DESCONHECIDO":
    cenarios.append(("brasil_normalizado", secoes[~secoes.SG_UF.eq("ZZ")], "faixa_brasilia"))
for nome, base, faixa in cenarios:
    if base.empty:
        continue
    rotulo_cenario = ROTULO if nome == "arquivo_todo_recorte" else ("Exterior; relógio local" if nome == "exterior_relogio_local" else ("Brasil; exclui exterior" if set(UFS_BR) <= set(base.SG_UF.unique()) else "Brasil no recorte: " + ", ".join(sorted(base.SG_UF.unique()))))
    tab = agregar(base, [faixa]); tabelas_horario[nome] = tab
    tab.to_csv(RAIZ / "tabelas" / f"horarios_{nome}.csv", index=False)
    display(Markdown(f"**{nome} — {rotulo_cenario}**")); display(tab)
    x = np.arange(len(tab))
    fig, ax = plt.subplots(figsize=(11, 5.5))
    for i, (n, nome_candidato) in enumerate(CANDIDATOS.items()):
        ax.bar(x + (i - 0.5) * 0.36, tab["participacao_" + n].astype(float), width=0.36, label=nome_candidato, color=CORES[n])
    ax.set(xticks=x, xticklabels=[str(v).replace(" a ", "\na ") for v in tab[faixa]], ylim=(0, 1), ylabel="Participação nos votos válidos", title=f"Votação por encerramento das seções\n{nome} | {rotulo_cenario}")
    ax.yaxis.set_major_formatter(PercentFormatter(1)); ax.legend(); salvar_fig("horarios_" + nome)
    agregar(base, ["SG_UF", faixa]).to_csv(RAIZ / "tabelas" / f"horarios_por_uf_{nome}.csv", index=False)
    campo = "segundos_brasilia" if faixa == "faixa_brasilia" else "segundos_arquivo"
    agregar(base[base[campo].ge(7200)], ["SG_UF"]).to_csv(RAIZ / "tabelas" / f"a_partir19_por_uf_{nome}.csv", index=False)
    agregar(base[base[campo].gt(7200)], ["SG_UF"]).to_csv(RAIZ / "tabelas" / f"apos19_estrito_por_uf_{nome}.csv", index=False)
    sensibilidade = []
    for minuto in range(0, 121, 15):
        for grupo, mask in [("até_corte", base[campo].le(minuto * 60)), ("após_corte", base[campo].gt(minuto * 60))]:
            linha = {"corte_brasilia_ou_arquivo": f"{17 + minuto//60:02}:{minuto%60:02}", "grupo": grupo, "secoes": int(mask.sum())}
            validos = int(base.loc[mask, "votos_validos"].sum())
            for n in CANDIDATOS:
                votos_n = int(base.loc[mask, "cand_" + n].sum())
                linha["votos_" + n] = votos_n; linha["participacao_" + n] = votos_n / validos if validos else np.nan
            sensibilidade.append(linha)
    pd.DataFrame(sensibilidade).to_csv(RAIZ / "tabelas" / f"sensibilidade_cortes_{nome}.csv", index=False)
    sens = pd.DataFrame(sensibilidade)
    fig, ax = plt.subplots(figsize=(10, 5))
    atras = sens[sens.grupo.eq("após_corte")]
    for n, candidato in CANDIDATOS.items():
        ax.plot(atras.corte_brasilia_ou_arquivo, atras["participacao_" + n], marker="o", color=CORES[n], label=candidato)
    ax.set(ylim=(0, 1), xlabel="Corte de encerramento", ylabel="Participação nos votos válidos", title=f"Seções encerradas após cada corte\n{nome} | {rotulo_cenario}")
    ax.yaxis.set_major_formatter(PercentFormatter(1)); ax.legend(); salvar_fig("sensibilidade_" + nome)

## 8. Biometria, distribuição e correlações
Taxa = contador biométrico / comparecimento. Faixas: 0–5%, >5–20%, >20–30%, >30–50%, >50–100%.
São calculadas correlações de Pearson e Spearman por seção, nacionais e por UF, e correlações com as variáveis centradas na média do município. O ajuste municipal reduz diferenças de composição geográfica, mas não produz uma estimativa causal. Não há pressuposto de distribuição gaussiana dos percentuais eleitorais.

In [ ]:
secoes["taxa_biometrica"] = (secoes.QT_ELEI_BIOM_SEM_HABILITACAO / secoes.QT_COMPARECIMENTO.replace(0, pd.NA)).astype(float)
secoes["faixa_biometrica"] = pd.cut(secoes.taxa_biometrica, bins=[-0.000001, .05, .20, .30, .50, 1], labels=["0–5%", ">5–20%", ">20–30%", ">30–50%", ">50–100%"])
tab_bio = agregar(secoes, ["faixa_biometrica"])
tab_bio.to_csv(RAIZ / "tabelas" / "biometria_faixas.csv", index=False); display(tab_bio)
agregar(secoes, ["SG_UF", "faixa_biometrica"]).to_csv(RAIZ / "tabelas" / "biometria_faixas_uf.csv", index=False)
if RELOGIO_ARQUIVO != "DESCONHECIDO":
    agregar(secoes[~secoes.SG_UF.eq("ZZ")], ["faixa_biometrica", "faixa_brasilia"]).to_csv(RAIZ / "tabelas" / "biometria_x_horario_brasilia.csv", index=False)
correlacoes = []
def registrar_correlacao(base, campo_y, contexto, numero, metodo):
    d = base[["taxa_biometrica", campo_y, "SG_UF", "CD_MUNICIPIO"]].dropna().copy()
    if metodo == "centrada_municipio":
        tamanhos = d.groupby(["SG_UF", "CD_MUNICIPIO"])[campo_y].transform("size")
        d = d[tamanhos.ge(2)]
        medias = d.groupby(["SG_UF", "CD_MUNICIPIO"])[["taxa_biometrica", campo_y]].transform("mean")
        d[["taxa_biometrica", campo_y]] = d[["taxa_biometrica", campo_y]] - medias
    linha = {"contexto": contexto, "numero": numero, "candidato": CANDIDATOS[numero], "metodo": metodo, "n_secoes": len(d), "coeficiente": np.nan}
    if len(d) >= MINIMO_SECOES_CORRELACAO and d.taxa_biometrica.nunique() > 1 and d[campo_y].nunique() > 1:
        func = stats.spearmanr if metodo == "spearman" else stats.pearsonr
        linha["coeficiente"] = float(func(d.taxa_biometrica.astype(float), d[campo_y].astype(float)).statistic)
    correlacoes.append(linha)
for n, nome in CANDIDATOS.items():
    y = "participacao_secao_" + n
    secoes[y] = secoes["cand_" + n] / secoes.votos_validos.replace(0, np.nan)
    for minimo in MINIMOS_VOTOS_VALIDOS:
        base = secoes[secoes.votos_validos.ge(minimo)]
        for metodo in ["pearson", "spearman", "centrada_municipio"]:
            registrar_correlacao(base, y, f"recorte_min_validos_{minimo}", n, metodo)
    for uf, base in secoes[secoes.votos_validos.gt(0)].groupby("SG_UF"):
        for metodo in ["pearson", "spearman"]:
            registrar_correlacao(base, y, uf, n, metodo)
    base = secoes[["taxa_biometrica", y]].dropna()
    if len(base):
        fig, ax = plt.subplots(figsize=(9, 5.5))
        h = ax.hexbin(base.taxa_biometrica.astype(float), base[y].astype(float), gridsize=40, mincnt=1, bins="log", cmap="Blues")
        fig.colorbar(h, ax=ax, label="Número de seções (escala log)")
        ax.set(xlim=(0, 1), ylim=(0, 1), xlabel="Contador biométrico / comparecimento", ylabel="Participação nos votos válidos", title=f"Biometria e votação: {nome}\n{ROTULO}; n={len(base):,} seções com dados")
        ax.xaxis.set_major_formatter(PercentFormatter(1)); ax.yaxis.set_major_formatter(PercentFormatter(1)); salvar_fig("biometria_" + n)
    fig, ax = plt.subplots(figsize=(9, 4.8))
    ax.hist(secoes[y].dropna().astype(float), bins=np.linspace(0, 1, 41), color=CORES[n], edgecolor="white")
    ax.set(xlim=(0, 1), xlabel="Participação nos votos válidos da seção", ylabel="Seções", title=f"Distribuição da votação: {nome}\n{ROTULO}")
    ax.xaxis.set_major_formatter(PercentFormatter(1)); salvar_fig("distribuicao_" + n)
pd.DataFrame(correlacoes).to_csv(RAIZ / "tabelas" / "correlacoes.csv", index=False)
display(pd.DataFrame(correlacoes).head(12))

## 9. Unanimidade e supermaiorias
Os critérios usam votos válidos e são aplicados simetricamente aos dois candidatos. Seções sem votos válidos ficam fora. A sensibilidade por tamanho ajuda a identificar o efeito de seções pequenas. A base de BU, sozinha, não identifica a presença de fiscais nem classifica o local como rural.

In [ ]:
unanimidade = []
for minimo in MINIMOS_VOTOS_VALIDOS:
    b = secoes[secoes.votos_validos.ge(minimo)]
    for n, nome in CANDIDATOS.items():
        unanimidade.append({"numero": n, "candidato": nome, "minimo_validos": minimo, "secoes_elegiveis": len(b), "secoes_100pct": int(b["cand_" + n].eq(b.votos_validos).sum()), "secoes_acima90pct": int(b["cand_" + n].gt(LIMIAR_SUPERMAIORIA * b.votos_validos).sum())})
tab_unanimidade = pd.DataFrame(unanimidade)
display(tab_unanimidade); tab_unanimidade.to_csv(RAIZ / "tabelas" / "unanimidade_sensibilidade.csv", index=False)
razoes = []
a, b = list(CANDIDATOS)
for minimo in MINIMOS_VOTOS_VALIDOS:
    t = tab_unanimidade[tab_unanimidade.minimo_validos.eq(minimo)].set_index("numero")
    for medida in ["secoes_100pct", "secoes_acima90pct"]:
        numerador, denominador = int(t.loc[a, medida]), int(t.loc[b, medida])
        razoes.append({"minimo_validos": minimo, "medida": medida, "numerador_candidato": CANDIDATOS[a], "denominador_candidato": CANDIDATOS[b], "numerador": numerador, "denominador": denominador, "razao": numerador / denominador if denominador else np.nan})
pd.DataFrame(razoes).to_csv(RAIZ / "tabelas" / "unanimidade_razoes.csv", index=False)
for n in CANDIDATOS:
    b = secoes[secoes.votos_validos.gt(0) & secoes["cand_" + n].gt(LIMIAR_SUPERMAIORIA * secoes.votos_validos)]
    b.to_csv(RAIZ / "tabelas" / f"secoes_acima90_{n}.csv", index=False)
    agregar(b, ["SG_UF"]).to_csv(RAIZ / "tabelas" / f"acima90_por_uf_{n}.csv", index=False)

## 10. Lei de Benford — diagnóstico exploratório
A distribuição esperada do primeiro dígito é `log10(1 + 1/d)`. Contagens de votos por seção têm limites e seleção geográfica; não se presume que devam seguir Benford. O qui-quadrado, seu p-valor e o desvio absoluto médio são descritivos sob essa referência, sem classificação automática de fraude. Resultados por UF ajudam a mostrar a heterogeneidade.

In [ ]:
benford_resumo, benford_digitos = [], []
def calcular_benford(valores, contexto, n):
    x = valores[valores.gt(0)].astype("int64")
    digitos = x.astype(str).str[0].astype(int)
    obs = digitos.value_counts().reindex(range(1, 10), fill_value=0).sort_index().to_numpy()
    teorica = np.log10(1 + 1 / np.arange(1, 10))
    tamanho = int(obs.sum())
    freq = obs / tamanho if tamanho else np.full(9, np.nan)
    chi, p = np.nan, np.nan
    if tamanho >= MINIMO_BENFORD and (tamanho * teorica >= 5).all():
        chi, p = stats.chisquare(obs, f_exp=tamanho * teorica)
    benford_resumo.append({"contexto": contexto, "numero": n, "n_secoes_voto_positivo": tamanho, "chi2_referencia": chi, "p_referencia": p, "desvio_absoluto_medio": np.mean(np.abs(freq - teorica))})
    for d, o, f, e in zip(range(1, 10), obs, freq, teorica):
        benford_digitos.append({"contexto": contexto, "numero": n, "digito": d, "contagem": int(o), "frequencia": f, "benford": e})
    return freq, teorica
for n, nome in CANDIDATOS.items():
    obs, teorica = calcular_benford(secoes["cand_" + n], "recorte", n)
    fig, ax = plt.subplots(figsize=(9, 4.8))
    ax.bar(np.arange(1, 10), obs, color=CORES[n], label="Observado")
    ax.plot(np.arange(1, 10), teorica, "o--", color="#333333", label="Referência Benford")
    ax.set(xticks=range(1, 10), xlabel="Primeiro dígito", ylabel="Frequência", title=f"Primeiro dígito dos votos por seção: {nome}\n{ROTULO}; apenas votos positivos")
    ax.set_ylim(bottom=0); ax.yaxis.set_major_formatter(PercentFormatter(1)); ax.legend(); salvar_fig("benford_" + n)
    for uf, b in secoes.groupby("SG_UF"):
        calcular_benford(b["cand_" + n], uf, n)
pd.DataFrame(benford_resumo).to_csv(RAIZ / "tabelas" / "benford_resumo.csv", index=False)
pd.DataFrame(benford_digitos).to_csv(RAIZ / "tabelas" / "benford_digitos.csv", index=False)
display(pd.DataFrame(benford_resumo).head(10))

## 11. Alegado × calculado
Os nomes são localizados na base, sem presumir seus números. Concordância numérica não valida a interpretação do relatório. Diferenças de população, definição ou fuso aparecem como limitação. A tolerância de 0,005 ponto percentual serve apenas para comparar um percentual publicado com duas casas decimais.

In [ ]:
def achar_candidato(fragmento):
    achados = [n for n, nome in CANDIDATOS.items() if fragmento in normalizar_nome(nome)]
    return achados[0] if len(achados) == 1 else None
lula, flavio = achar_candidato("LULA"), achar_candidato("FLAVIO")
comparacoes = []
def comparar(medida, alegado, calculado, observacao="", tolerancia=0):
    disponivel = calculado is not None and not pd.isna(calculado)
    diferenca = float(calculado) - float(alegado) if disponivel else np.nan
    if not NACIONAL_COMPLETO:
        situacao = "RECORTE PARCIAL; não compara o total nacional"
    elif not disponivel:
        situacao = "INDETERMINADO"
    else:
        situacao = "COINCIDE" if abs(diferenca) <= tolerancia + 1e-12 else "DIVERGE"
    comparacoes.append({"medida": medida, "alegado": alegado, "calculado": calculado, "diferenca": diferenca, "situacao": situacao, "observacao": observacao})
comparar("Seções com BU presidencial apurado", ALEGACOES["secoes"], len(secoes), "Seções agregadas não são expandidas.")
for campo, label in [("QT_COMPARECIMENTO", "comparecimento"), ("votos_totais", "votos totais"), ("votos_validos", "votos válidos")]:
    comparar('Expressão ambígua "votos apurados" versus ' + label, ALEGACOES["votos_apurados_termo_ambiguo"], int(secoes[campo].sum()))
bio_total = int(secoes.QT_ELEI_BIOM_SEM_HABILITACAO.sum()) if secoes.QT_ELEI_BIOM_SEM_HABILITACAO.notna().all() else None
comparar("Contador biométrico oficial", ALEGACOES["contador_biometrico"], bio_total, "Eleitores com biometria não habilitados por ela; não equivale automaticamente a votos irregulares.")
totais = {n: int(secoes["cand_" + n].sum()) for n in CANDIDATOS}
margem = abs(totais[list(CANDIDATOS)[0]] - totais[list(CANDIDATOS)[1]])
comparar("Margem absoluta entre candidatos", ALEGACOES["margem_absoluta"], margem)
for numero, fragmento in [(lula, "lula"), (flavio, "flavio")]:
    if numero is not None:
        b = tab_unanimidade[(tab_unanimidade.numero == numero) & (tab_unanimidade.minimo_validos == 1)].iloc[0]
        comparar("Seções unânimes — " + CANDIDATOS[numero], ALEGACOES["unanimes_" + fragmento], int(b.secoes_100pct))
        comparar("Seções acima de 90% — " + CANDIDATOS[numero], ALEGACOES["acima90_" + fragmento], int(b.secoes_acima90pct))
if flavio:
    for modo, campo, base in [("relógio do arquivo; Brasil + exterior", "segundos_arquivo", secoes), ("Brasília; apenas Brasil", "segundos_brasilia", secoes[~secoes.SG_UF.eq("ZZ")])]:
        b = base[base[campo].le(0)]
        share = int(b["cand_" + flavio].sum()) / int(b.votos_validos.sum()) if b.votos_validos.sum() else None
        comparar("Participação de Flávio até 17h — " + modo, ALEGACOES["participacao_flavio_ate17"], share, "Corte por encerramento da seção; o cenário Brasil exclui exterior e difere do universo nacional alegado.", tolerancia=.00005)
tab_comparacao = pd.DataFrame(comparacoes)
display(tab_comparacao)
tab_comparacao.to_csv(RAIZ / "tabelas" / "alegado_calculado.csv", index=False)
resumo = {"escopo": ROTULO, "nacional_particoes_completas": NACIONAL_COMPLETO, "secoes": len(secoes), "comparecimento": int(secoes.QT_COMPARECIMENTO.sum()), "votos_validos": int(secoes.votos_validos.sum()), "votos_por_candidato": totais, "margem_absoluta": margem, "contador_biometrico": bio_total, "razao_contador_margem": bio_total/margem if bio_total is not None and margem else None, "reconciliacao_externa_informada": bool(REFERENCIA_OFICIAL), "relogio_arquivo_hipotese": RELOGIO_ARQUIVO, "horarios_invalidos": int(invalidos_horario.sum())}
json_save("resumo_calculado.json", resumo)
display(Markdown(f"**Resultados calculados no recorte:** {len(secoes):,} seções; {resumo['votos_validos']:,} votos válidos. Consulte as tabelas para as definições e diferenças. A razão entre contador biométrico e margem é uma comparação de magnitudes, sem atribuição de votos a candidatos."))

## 12. Limites temporais por candidato

A contagem exata até o instante escolhido pode ser impossível com os BUs. As seções encerradas depois do corte e as sem horário confiável permanecem no universo e contribuem para a incerteza.

In [ ]:
# @title 12. Limites de votos até o horário escolhido — todas as seções
CORTE_TEMPORAL = pd.Timestamp(f"{DATA_ELEICAO} {HORARIO_CORTE}")
if CORTE_TEMPORAL.strftime("%Y-%m-%d %H:%M:%S") != f"{DATA_ELEICAO} {HORARIO_CORTE}":
    raise ValueError("Informe o corte como HH:MM:SS, sem fuso ou frações de segundo.")

def apurar_limites_temporais(base, candidatos, corte):
    """Limites por candidato; o BU final não contém a cronologia dos votos.

    Uma seção encerrada até o corte permite afirmar que todos os seus votos
    foram registrados até ele. Nas outras, o BU não permite repartir os votos
    por horário. Horários ausentes, inclusive os do exterior sem conversão,
    permanecem desconhecidos e suas seções continuam no universo analisado.
    """
    necessarias = ["encerramento_brasilia"] + ["cand_" + str(n) for n in candidatos]
    faltam = sorted(set(necessarias) - set(base.columns))
    if faltam:
        raise ValueError(f"Execute primeiro as células anteriores. Colunas ausentes: {faltam}")
    if base.empty:
        raise ValueError("Não há seções carregadas; uma base vazia não equivale a zero votos.")
    horarios = pd.to_datetime(base["encerramento_brasilia"], errors="raise")
    comprovadamente_antes = horarios.notna() & horarios.le(corte)
    ainda_indeterminadas = ~comprovadamente_antes
    linhas = []
    for numero, candidato in candidatos.items():
        valores = pd.to_numeric(base["cand_" + str(numero)], errors="raise")
        if valores.isna().any() or valores.lt(0).any() or valores.mod(1).ne(0).any():
            raise ValueError(f"{candidato}: contagens ausentes, negativas ou não inteiras.")
        minimo = int(valores.loc[comprovadamente_antes].sum())
        sem_distribuicao_horaria = int(valores.loc[ainda_indeterminadas].sum())
        maximo = minimo + sem_distribuicao_horaria
        exato = minimo if minimo == maximo else pd.NA
        linhas.append({
            "Candidato": candidato,
            "Votos finais em todas as seções": int(valores.sum()),
            "Mínimo comprovado até o corte": minimo,
            "Limite superior conservador": maximo,
            "Total exato até o corte": exato,
            "Situação": "EXATO" if minimo == maximo else "INDETERMINADO",
        })
    resultado = pd.DataFrame(linhas)
    resultado["Total exato até o corte"] = resultado["Total exato até o corte"].astype("Int64")
    cobertura = {
        "corte_brasilia": str(corte),
        "secoes_no_universo": len(base),
        "secoes_encerradas_ate_corte_inclusive": int(comprovadamente_antes.sum()),
        "secoes_encerradas_apos_corte": int(horarios.gt(corte).sum()),
        "secoes_sem_horario_brasilia_confiavel": int(horarios.isna().sum()),
    }
    return resultado, cobertura

resultado_corte, cobertura_corte = apurar_limites_temporais(secoes, CANDIDATOS, CORTE_TEMPORAL)
display(Markdown(f"### Corte inclusivo: {CORTE_TEMPORAL} — Brasília"))
display(Markdown(
    "Todas as seções carregadas permanecem no universo, inclusive as que fecharam depois do corte. "
    "O mínimo soma os votos das seções encerradas até o corte. O limite superior conservador "
    "é o total final do candidato. Esses limites não estimam a distribuição dos votos no tempo. "
    "INDETERMINADO não significa zero. Exterior sem conversão de fuso permanece indeterminado. "
    "Os limites dependem da hipótese de fuso escolhida; DESCONHECIDO produz limites triviais."
))
display(pd.DataFrame([cobertura_corte]))
display(resultado_corte)
sufixo_corte = HORARIO_CORTE.replace(":", "")
resultado_corte.to_csv(RAIZ / "tabelas" / f"limites_votos_ate_{sufixo_corte}.csv", index=False)
json_save(f"cobertura_corte_{sufixo_corte}.json", cobertura_corte)
display(Markdown(
    "O corte de 17h16 é uma escolha desta análise. "
    "[O TSE descreve 17h10 como estimativa de procedimentos de encerramento]"
    "(https://www.tse.jus.br/comunicacao/noticias/2026/Outubro/eleicoes-2026-mais-de-34-mil-votos-foram-registrados-apos-as-17h-no-2o-turno). "
    "Não se trata de um prazo universal de validade dos votos."
))


## 13. Exportar e baixar os resultados
O ZIP inclui os parâmetros, catálogo consultado, manifesto com hashes, dicionários, tabelas e gráficos. Não inclui os ZIPs brutos nem os Parquets, para limitar o tamanho; seus links e hashes permitem recuperá-los. O histórico alegado de publicação não pode ser estabelecido apenas pelo nome atual de um arquivo.
Alegações sobre fiscais, ruralidade, decisões judiciais ou atos de autoridades exigem fontes adicionais e ficam fora desta análise de BUs.

In [ ]:
import importlib.metadata
configuracao = {"data_eleicao": DATA_ELEICAO, "corte_temporal": HORARIO_CORTE, "corte_inclusivo": True, "ano": ANO, "turno": TURNO, "cargo": CARGO, "escopo": ESCOPO, "ufs_esperadas": UF_ESPERADAS, "ufs_processadas": sorted(secoes.SG_UF.unique()), "referencia_oficial": REFERENCIA_OFICIAL, "alegacoes": ALEGACOES, "relogio_arquivo": RELOGIO_ARQUIVO, "tamanho_bloco": TAMANHO_BLOCO, "minimos_validos": MINIMOS_VOTOS_VALIDOS, "semente": SEMENTE, "executado_em_utc": agora(), "python": platform.python_version(), "bibliotecas": {mod: importlib.metadata.version(mod) for mod in DEPENDENCIAS}}
json_save("parametros_execucao.json", configuracao)
secoes.to_parquet(RAIZ / "intermediarios" / "secoes_consolidadas.parquet", index=False)
texto_leia = f"""REPRODUÇÃO TSE 2026 — {ROTULO}
Execução: {configuracao['executado_em_utc']}
Fonte: {CATALOGO}
Definição biométrica: eleitores com biometria que não foram habilitados por meio dela.
Taxa: contador dividido por comparecimento, não por eleitores com cadastro biométrico.
Horário do arquivo: hipótese {RELOGIO_ARQUIVO}; veja os cenários e a conferência de aberturas.
Horários de encerramento não datam votos individuais.
Partições completas: {NACIONAL_COMPLETO}. Reconciliação independente informada: {bool(REFERENCIA_OFICIAL)}.
Contadores de seção não foram somados por candidato. Seções agregadas não foram expandidas.
Correlações e Benford são exploratórios e não estabelecem fraude.
Arquivo alegado_calculado.csv: o status de coincidência compara números e não confirma causalidade ou interpretação.
Gráficos usam todos os registros do recorte válido; não há amostragem gráfica de seções.
"""
(RAIZ / "LEIA-ME.txt").write_text(texto_leia, encoding="utf-8")
PACOTE = RAIZ / f"resultados_tse_{ANO}_{ESCOPO.lower()}.zip"
with zipfile.ZipFile(PACOTE, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(RAIZ.rglob("*")):
        if p.is_file() and p != PACOTE and not p.name.endswith(".zip") and "arquivos" not in p.relative_to(RAIZ).parts and "intermediarios" not in p.relative_to(RAIZ).parts:
            z.write(p, p.relative_to(RAIZ))
print("Concluído:", PACOTE)
print("No Colab, execute a célula opcional abaixo para baixar o ZIP.")
json_save("localizacao_resultados.json", {"pasta_execucao": RAIZ.name, "arquivo_zip": PACOTE.name})


In [ ]:
# @title Baixar o ZIP no Colab (opcional; execute ao final)
try:
    from google.colab import files
except ImportError:
    print("Fora do Colab, use o ZIP na pasta de resultados.")
else:
    files.download(str(PACOTE))
